# TissueSpectF on ColabChromosome-ordered spectral analysis of tissue transcriptomes, end to end, on afree Colab VM.**Read this before you start.** Colab gives you about 2 cores. The per-sample`maxt` stage took 41 minutes on 23 cores with 359 samples, which is roughly**8 hours here** and will outlive your session. The `condition` stage answersthe primary question at about 1/n of that cost, so the default run below skips`maxt`.What you give up by skipping it: the per-sample reproducibility figures(`pct_samples_significant`) and the Stouffer combination. What you keep: thecondition-level permutation test, which is the criterion that decides whichpeaks go downstream.Run **Runtime > Change runtime type > R** if the option is offered. Otherwisethe first cell installs R.

In [ ]:
# R runtime? Skip this cell. Python runtime? This installs R (~2 min).import subprocess, shutilif shutil.which("Rscript") is None:    subprocess.run("apt-get -qq update && apt-get -qq install -y r-base-core", shell=True)print(subprocess.run(["Rscript", "--version"], capture_output=True, text=True).stderr.strip())

## 1. Get the code

In [ ]:
!git clone --depth 1 https://github.com/Danpc11/TissueSpectF.git 2>/dev/null || echo "already cloned"%cd TissueSpectF!chmod +x tsf

## 2. Point the paths at Colab storage`/content` is wiped when the session ends. To keep results, mount Drive andpoint `TSF_RESULTS_DIR` there instead.

In [ ]:
import osos.environ["TSF_GEO_DIR"]     = "/content/data"os.environ["TSF_INTERIM_DIR"] = "/content/interim"os.environ["TSF_RESULTS_DIR"] = "/content/results"# Fewer permutations than the default: enough to conclude, cheap enough to finish.# The floor on q is n_chromosomes/(B+1), so B = 2000 keeps q reachable below 0.05.os.environ["TSF_CONDITION_B"] = "2000"# from google.colab import drive; drive.mount('/content/drive')# os.environ["TSF_RESULTS_DIR"] = "/content/drive/MyDrive/TissueSpectF/results"

## 3. Check the code works before spending any time on data

In [ ]:
!make test!./tsf selfcheck

## 4. Download the GEO inputsAbout 20 MB. Skips anything already present.

In [ ]:
!./tsf fetch!./tsf check

## 5. Run`--to=spectra` first: it is fast and tells you the grid coverage per datasetbefore you commit to the permutations.

In [ ]:
!./tsf run --to=spectra

Then the condition-level test and everything downstream. Expect roughly15-30 minutes on two cores. `--from=condition` is what skips the per-sample`maxt`; the stability stage falls back to the condition test on its own.

In [ ]:
!./tsf run --from=condition --log=/content/results/run.log

## 6. Look at what came outThe spectral window first: any peak sitting in the top 1% of the window is asampling artefact until shown otherwise.

In [ ]:
!./tsf window

In [ ]:
import pandas as pd, glob, osres = os.environ["TSF_RESULTS_DIR"]for f in sorted(glob.glob(f"{res}/*/condition/condition_significance_average_*.tsv")):    d = pd.read_csv(f, sep="\t")    hit = d[d.q_condition <= 0.05]    if len(hit):        print(os.path.basename(f))        print(hit[["chr","N","k","period","amplitude","p_condition",                   "q_condition","window_rank"]].head(10).to_string(index=False))        print()

## Optional: per-sample maxTOnly if you have a paid runtime with more cores, or you restrict the scope.One condition of one dataset is tractable:```!./tsf maxt GSE162694 --cond=F3```Then rerun `condition` to pick up the Stouffer column, and `stability`.

## Optional: the second gene universeThe default grid is protein-coding. To rerun with non-coding genes included,send it to a separate results tree so the two do not overwrite each other:```import osos.environ["TSF_GENE_UNIVERSE"] = "^(protein-coding|ncRNA)$"os.environ["TSF_RESULTS_DIR"]   = "/content/results_pc_nc"os.environ["TSF_INTERIM_DIR"]   = "/content/interim_pc_nc"!./tsf run --from=ingest --to=spectra!./tsf run --from=condition```A peak that survives both universes is robust to the definition; one that doesnot tells you the result depends on which genes you count.